In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
import emoji

In [6]:
df = pd.read_csv(r"C:\\Users\\User\\OneDrive\\Documents\\sentimenet analysis datasets\\Reviews.csv")

In [7]:

print(df.columns.tolist())

['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']


In [8]:
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [9]:
df = df[['Score', 'Text']]

In [10]:
df = df[df['Score'] !=3]
df['Sentiment'] = df['Score'].apply(lambda x: 1 if x >= 4 else 0)

In [11]:
df = df.drop(columns=['Score'])

In [12]:
print(df.shape)

(525814, 2)


In [13]:
df.head

<bound method NDFrame.head of                                                      Text  Sentiment
0       I have bought several of the Vitality canned d...          1
1       Product arrived labeled as Jumbo Salted Peanut...          0
2       This is a confection that has been around a fe...          1
3       If you are looking for the secret ingredient i...          0
4       Great taffy at a great price.  There was a wid...          1
...                                                   ...        ...
568449  Great for sesame chicken..this is a good if no...          1
568450  I'm disappointed with the flavor. The chocolat...          0
568451  These stars are small, so you can give 10-15 o...          1
568452  These are the BEST treats for training and rew...          1
568453  I am very satisfied ,product is as advertised,...          1

[525814 rows x 2 columns]>

In [14]:
df = df.sample(n=50000, random_state=42) #use only 50k row 

In [15]:
df = df.reset_index(drop=True) 

In [16]:
texts = df['Text'].dropna().astype(str)

In [17]:
print(df.shape)

(50000, 2)


In [18]:
print(df['Sentiment'].value_counts())

Sentiment
1    42147
0     7853
Name: count, dtype: int64


In [19]:
df.head()

,Text,Sentiment
0,This is a very high quality dog food with meat...,1
1,I love this cake mix and the other 3 mixes as ...,1
2,A nice strong brew. I am new to Keurig and hav...,1
3,I just found PB2 and PB2 with chocolate and I ...,1
4,Delightful mint tea as one would expect. Note ...,1


In [20]:
emote_pattern = r'((?:[:=;][oO\-]?[D\)\]\(\]/\\OpP]))' #this allow to remove emotes such as 
#:), :D, o_o etc

In [21]:
special_symbols_pattern = r'[^\w\s\.\,\!\?\'\"\-\@]' #removing special symbols that occur within the datasets

In [22]:
#understand how many types of emotes within the datasets, including emoticon and special symbols
emoji_counts = Counter()
emoticon_counts = Counter()
symbol_counts = Counter()
print("Scanning texts for emojis, emoticons, and symbols...")

Scanning texts for emojis, emoticons, and symbols...


In [23]:
for text in texts:
    extracting_emoji = [res['emoji'] for res in emoji.emoji_list(text)]
    emoji_counts.update(extracting_emoji)
    #emoji_list returns a lists of dicts with the emoji and its position
    emoticons = re.findall(emote_pattern, text)
    emoticon_counts.update(emoticons)
    #finding text that match emote_pattern
    clean_text = re.sub(emote_pattern, '', text)
    clean_text = emoji.replace_emoji(clean_text, replace='')
    #firstly, remove the emoticon and emojis from the text string temporary so that their component
    #symbols like :) or :D dont get counted as standalone symbols
    symbols = re.findall(special_symbols_pattern, clean_text)
    symbol_counts.update(symbols)

In [24]:
for char, count in emoji_counts.most_common(15):
    print(f"{char} : {count}")

® : 99


In [25]:
for char, count in emoticon_counts.most_common(15):
    print(f"{char} : {count}")

:/ : 1432
:) : 861
:-) : 170
:( : 112
;) : 100
:D : 53
=) : 45
;-) : 41
:-( : 36
:o) : 16
:P : 14
:-D : 9
=( : 8
=p : 8
=D : 6


In [26]:
for char, count in symbol_counts.most_common(20):
    print(f"{char} : {count}")

/ : 67972
> : 57836
< : 57700
) : 17644
( : 17262
: : 4699
& : 3992
$ : 3458
; : 2875
% : 2145
* : 2040
= : 1902
+ : 829
~ : 572
[ : 534
] : 526
# : 309
` : 67
^ : 31
} : 26


In [27]:
#elaborate what happenings

In [28]:
def final_clean_for_amazon(text):
    #remove html tags
    text = re.sub(r'<.*?>', ' ', text)
    #convert specific sentiment heavy emoticon to words
    #this ensures "good :)" stays positive even after stripping symbols
    emoticons_map = {
        r':\)': ' happyemoticon ',
        r':-\)': ' happyemoticon ',
        r':D': ' happyemoticon ',
        r':\(': ' sademoticon ',
        r':-\(': ' sademoticon ',
        r':/': ' skepticalemoticon '
    }
    for pattern, word in emoticons_map.items():
        text = re.sub(pattern, word, text)

        #handle the '&' and other entities
        text = re.sub(r'&[\w#]+;', ' ', text)
        #remove the leftover symbols/numbers but keep our new emoticon words
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        #lowercase and remove extra whitespace
        text = text.lower().strip()
        text = re.sub(r'\s+', ' ', text)
        return text

In [32]:
sample = "The coffee was great! :) <br /> But the price was too high $$. :/"
print(final_clean_for_amazon(sample))

the coffee was great happyemoticon but the price was too high
